# Running this experiment

See README.md for setup, required inputs, and limitations. Run cells in order from a fresh kernel. Saved outputs have been cleared; historical metrics are not newly verified results. Environment installation and external training commands are manual setup steps.

In [ ]:
# Override these paths for your local datasets. PROJECT_DATA_DIR changes their base.
INPUT_PATHS = {}

# Import Libraries

In [ ]:
# Keep warnings visible so deprecated APIs and fit failures can be diagnosed.


# Standard scikit-learn works across platforms without an optional Intel patch.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# linear svc

In [ ]:
TrainFeatures = pd.read_csv("X_train_scaled_balanced.csv", header="infer")
TestFeatures = pd.read_csv("X_test_scaled.csv", header="infer")

TrainOutcome = pd.read_csv("y_train_balanced.csv", header="infer")
TestOutcome = pd.read_csv("y_test.csv", header="infer")

In [ ]:
TrainOutcome

In [ ]:
print(TrainOutcome["Diabetes"].value_counts())
print(TestOutcome["Diabetes"].value_counts())

# class countplot for cheking imbalance data and data volume

In [ ]:
sns.countplot(x="Diabetes", data=TrainOutcome)
print("Train outcome class count")
# Show the plot
plt.show()

In [ ]:
sns.countplot(x="Diabetes", data=TestOutcome)
print("test outcome class count")
# Show the plot
plt.show()

In [ ]:
# Calculate the correlation matrix for Data
correlation_matrix = TrainFeatures.corr(numeric_only=True)
# Create a heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

# default linear svc hyperparameter on data

In [ ]:
print(TrainFeatures.shape)
print(TestFeatures.shape)
ychart = np.array([425532, 106384])
mylabels = ["train dataset size: 425532", "test dataset size: 106384"]
plt.pie(ychart, labels=mylabels)
plt.show()

# hyperparameter : class sklearn.svm.LinearSVC(penalty='l2', loss='squared_hinge', *, dual='auto', tol=0.0001, C=1.0, multi_class='ovr', fit_intercept=True, intercept_scaling=1, class_weight=None, verbose=0, random_state=None, max_iter=1000)

In [ ]:
X_train = TrainFeatures
y_train = TrainOutcome
X_test = TestFeatures
y_test = TestOutcome
lsvc = LinearSVC(verbose=0)
lsvc.fit(X_train, y_train)
score = lsvc.score(X_train, y_train)
print("Score: ", score)

In [ ]:
y_pred = lsvc.predict(X_test)
print("svm Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
X_train

In [ ]:
cc = [0.0001, 0.001, 0.01, 0.1, 1, 10]
for i in cc:
    lsvc = LinearSVC(C=i)
    lsvc.fit(X_train, y_train)
    score = lsvc.score(X_train, y_train)
    print("train acc: ", score)
    print(i)

In [ ]:
cc = ["l1", "l2"]
for i in cc:
    lsvc = LinearSVC(C=0.0001, penalty=i, dual=False)
    lsvc.fit(X_train, y_train)
    score = lsvc.score(X_train, y_train)
    print("train acc: ", score)
    print(i)

In [ ]:
cc = ["hinge", "squared_hinge"]
for i in cc:
    lsvc = LinearSVC(C=0.0001, loss=i)
    lsvc.fit(X_train, y_train)
    score = lsvc.score(X_train, y_train)
    print("train acc: ", score)
    print(i)

In [ ]:
cc = [False, True]
for i in cc:
    lsvc = LinearSVC(C=0.0001, dual=i)
    lsvc.fit(X_train, y_train)
    score = lsvc.score(X_train, y_train)
    print("train acc: ", score)
    print(i)

In [ ]:
cc = [1000, 5000, 10000, 15000]
for i in cc:
    lsvc = LinearSVC(C=0.0001, max_iter=i)
    lsvc.fit(X_train, y_train)
    score = lsvc.score(X_train, y_train)
    print("train acc: ", score)
    print(i)

In [ ]:
lsvc = LinearSVC(C=0.1, max_iter=1000, loss="hinge", penalty="l2")
lsvc.fit(X_train, y_train)
score = lsvc.score(X_train, y_train)

print("train acc: ", score)

y_pred = lsvc.predict(X_test)
print("svm test Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
from sklearn.decomposition import PCA

cc = TrainFeatures.columns
# Initiating PCA to reduce dimentions aka features to 3
pca = PCA(n_components=3)
pca.fit(TrainFeatures)
PCA_ds1 = pd.DataFrame(pca.transform(TrainFeatures), columns=(["col1", "col2", "col3"]))
PCA_ds1.describe().T

# using pca on normalized dataset

In [ ]:
lsvc = LinearSVC(verbose=0)
lsvc.fit(PCA_ds1, y_train)
score = lsvc.score(PCA_ds1, y_train)
print("train acc: ", score)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

model = LinearSVC(random_state=42)
# Define hyperparameter distribution
param_dist = {
    "C": [0.0001, 0.001, 0.01, 0.1, 1, 10],
}
# Perform random search
random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=1000,
    cv=5,
    scoring="accuracy",
    random_state=42,
)
random_search.fit(X_train, y_train)
# Report best score and parameters
print(f"Best score: {random_search.best_score_:.3f}")
print(f"Best parameters: {random_search.best_params_}")
# Evaluate on test set
best_model = random_search.best_estimator_
accuracy = best_model.score(X_test, y_test)
print(f"Test set accuracy: {accuracy:.3f}")

In [ ]:
print(f"Test set accuracy: {accuracy:.5f}")

In [ ]:
scores = [
    ["best C with RandomizedSearchCV train accuracy", 0.716],
    ["best C with RandomizedSearchCV test accuracy", 0.68808],
    ["train accuracy after using PCA", 0.6662],
    ["test accuracy after using PCA", 0.66400],
]


plot_df = pd.DataFrame(scores, columns=["METHOD", "ACCURACY"])
plt.figure(figsize=(10, 6))
sns.barplot(x="METHOD", y="ACCURACY", data=plot_df, color="blue")
plt.show()
display(plot_df)

In [ ]:
# Calculate the correlation matrix for Data
dfN = TrainFeatures
dfN["class"] = TrainOutcome
dfN

# show correlation with class

In [ ]:
correlation_matrix = dfN.corr(numeric_only=True)

# Create a heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
dfN_encoded = dfN.drop(
    [
        "FriedPotato_Consumption",
        "Green_Vegetables_Consumption",
        "Fruit_Consumption",
        "Sex",
    ],
    axis=1,
)

xnew = dfN_encoded.drop("class", axis=1)
ynew = dfN_encoded["class"]
X_train5 = xnew
y_train5 = ynew
TestFeatures = pd.read_csv("X_test_scaled.csv", header="infer")
TestFeatures = TestFeatures.drop(
    [
        "FriedPotato_Consumption",
        "Green_Vegetables_Consumption",
        "Fruit_Consumption",
        "Sex",
    ],
    axis=1,
)
x_test5 = TestFeatures
y_test5 = TestOutcome
# y_train5 = np.squeeze(y_train5)
lsvc = LinearSVC(verbose=0)
lsvc.fit(X_train5, y_train5)
score = lsvc.score(X_train5, y_train5)
print("train acc: ", score)

y_pred = lsvc.predict(x_test5)
print("svm test Accuracy:", accuracy_score(y_test5, y_pred))

In [ ]:
correlation_matrix = dfN_encoded.corr(numeric_only=True)

# Create a heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
dfN_encoded

In [ ]:
dfN_encoded = dfN_encoded.drop(
    ["General_Health", "Skin_Cancer", "Other_Cancer", "Depression", "Smoking_History"],
    axis=1,
)
xnew = dfN_encoded.drop("class", axis=1)
ynew = dfN_encoded["class"]
X_train5 = xnew
y_train5 = ynew
x_test5 = x_test5.drop(
    ["General_Health", "Skin_Cancer", "Other_Cancer", "Depression", "Smoking_History"],
    axis=1,
)

lsvc = LinearSVC(verbose=0)
lsvc.fit(X_train5, y_train5)
score = lsvc.score(X_train5, y_train5)
print("train acc: ", score)

y_pred = lsvc.predict(x_test5)
print("svm test Accuracy:", accuracy_score(y_test5, y_pred))

In [ ]:
scores = [
    ["best C with RandomizedSearchCV train accuracy", 0.716],
    ["best C with RandomizedSearchCV test accuracy", 0.68808],
    ["train accuracy after using PCA", 0.6662],
    ["test accuracy after using PCA", 0.66400],
    ["train accuracy after drop 4 unimportant feature", 0.71503],
    ["test accuracy after drop 4 unimportant feature", 0.68774],
    ["train accuracy after second droping features", 0.71262],
    ["test accuracy after second droping features", 0.68482],
]


plot_df = pd.DataFrame(scores, columns=["METHOD", "ACCURACY"])
plt.figure(figsize=(10, 6))
sns.barplot(x="METHOD", y="ACCURACY", data=plot_df, color="blue")
plt.xticks(rotation="vertical")
plt.show()
display(plot_df)

In [ ]:
print("best test accuracy: 0.68808 for Tuned linearSVC by RandomizedSearchCV")

In [ ]:
TrainFeatures = pd.read_csv("X_train_scaled_balanced.csv", header="infer")
TestFeatures = pd.read_csv("X_test_scaled.csv", header="infer")

TrainOutcome = pd.read_csv("y_train_balanced.csv", header="infer")
TestOutcome = pd.read_csv("y_test.csv", header="infer")
X_train = TrainFeatures
y_train = TrainOutcome
X_test = TestFeatures
y_test = TestOutcome
lsvc = LinearSVC(C=0.001, max_iter=1000, loss="hinge", penalty="l2")
lsvc.fit(X_train, y_train)
score = lsvc.score(X_train, y_train)
print("train acc: ", score)
y_pred = lsvc.predict(X_test)
print("svm test Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn import metrics

confusion_matrix = confusion_matrix(y_test, y_pred)
cm_display = metrics.ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix, display_labels=[0, 1]
)

cm_display.plot()
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc

# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred)
roc_auc = auc(fpr, tpr)
# Plot the ROC curve
plt.figure()
plt.plot(fpr, tpr, label="ROC curve (area = %0.2f)" % roc_auc)
plt.plot([0, 1], [0, 1], "k--", label="No Skill")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve for linearSVC Classification")
plt.legend()
plt.show()

In [ ]:
# Extract feature coefficients
feature_coefficients = lsvc.coef_
for i, coeff in enumerate(feature_coefficients):
    print(f"Feature {i + 1}: {coeff}")

In [ ]:
feature_coefficients = feature_coefficients.T
feature_coefficients[0]

In [ ]:
xxx = X_train.columns
xxx.shape

In [ ]:
print("SVC feature importance")
global_importances = pd.Series(feature_coefficients[0], index=xxx)
global_importances = global_importances.iloc[0:11]
global_importances.sort_values(ascending=True, inplace=True)
global_importances.plot.barh(color="blue")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Feature Importance in SVC")